DataLake (Deltalake) + Lakehouse (Deltatables) - using Delta format (parquet+snappy+delta log)
Delta Lake is an open-source storage framework that brings reliability, ACID transactions, and performance to data lakes. It sits on top of Parquet files and is most commonly used with Apache Spark and Databricks.
Delta Lake is built natively on top of Apache Parquet file. will not support ORC
Delta Lake & Deltalakhouse is the Core/Analytical storage layer behind Bronze–Silver–Gold (medallion) architectures.

In [0]:
spark.sql(f"create catalog if not exists lakehousecat1")
spark.sql(f"create schema if not exists lakehousecat1.deltadb")
spark.sql(f"create volume if not exists lakehousecat1.deltadb.deltalake")

In [0]:
df = spark.read.csv("/Volumes/lakehousecat1/deltadb/deltalake/drugsinfo.csv",header=True,inferSchema=True)

# saving as parquet
df.write.format("parquet").mode("overwrite").save("/Volumes/lakehousecat1/deltadb/deltalake/targetparquet")
#saving as delta file
df.write.format("delta").mode("overwrite").save("/Volumes/lakehousecat1/deltadb/deltalake/targetdelta")
# by default saving as delta table
df.write.saveAsTable("lakehousecat1.deltadb.drugstbl",mode="overwrite")

spark.sql("select * from lakehousecat1.deltadb.drugstbl").show()



In [0]:
# taking count from file

spark.read.format("parquet").load("/Volumes/lakehousecat1/deltadb/deltalake/targetparquet").count()
spark.read.format("delta").load("/Volumes/lakehousecat1/deltadb/deltalake/targetdelta").count()

In [0]:
%sql

-- description of table

describe formatted lakehousecat1.deltadb.drugstbl;

In [0]:
%sql
 -- execution plan

explain select * from lakehousecat1.deltadb.drugstbl;

In [0]:
# schema evolution

df.write.option("mergeSchema","true").format("delta").mode("overwrite").saveAsTable("lakehousecat1.deltadb.drugstbl")

spark.sql("select * from lakehousecat1.deltadb.drugstbl").show()
